# 04 · Dictionaries (and 🆕 `frozendict`)

**Module 03 – Data Structures** · Estimated time: 60 min

### 🎯 What you will learn
- Key → value mappings; creating dicts in 5 ways
- Accessing safely with `get()`; adding, updating, deleting
- Every important method: `keys values items get setdefault update pop popitem clear copy fromkeys`
- Dictionary views (they're *live*!)
- Iterating, sorting and nesting dictionaries
- Merging with `|` and `|=` (3.9+)
- Common patterns: counting, grouping, lookup tables
- 🆕 `frozendict`, the immutable dictionary (Python 3.15)

## 1. What is a dictionary?

A dictionary stores **key-value pairs**. You look things up by **key** (not by position), and lookups are extremely fast (hashing, like sets).

- Keys must be **unique** and **hashable** (str, int, tuple, ...)
- Values can be **anything**
- Since Python 3.7, dicts **remember insertion order**

In [1]:
student = {"name": "Alice", "age": 21, "grade": "A"}
print(student)
print(student["name"])
print(len(student))

{'name': 'Alice', 'age': 21, 'grade': 'A'}
Alice
3


### Five ways to create a dictionary

In [2]:
d1 = {"a": 1, "b": 2}                              # literal
d2 = dict(a=1, b=2)                                # keyword arguments
d3 = dict([("a", 1), ("b", 2)])                    # list of pairs
d4 = dict(zip(["a", "b"], [1, 2]))                 # zip two lists
d5 = {k: v for k, v in [("a", 1), ("b", 2)]}       # comprehension
print(d1 == d2 == d3 == d4 == d5)
print(dict.fromkeys(["x", "y", "z"], 0))           # same value for every key

True
{'x': 0, 'y': 0, 'z': 0}


## 2. Accessing values

In [3]:
student["email"]          # ❌ KeyError when the key doesn't exist

KeyError: 'email'

In [4]:
print(student.get("email"))                    # None instead of an error
print(student.get("email", "not provided"))    # with a default
print("email" in student, "name" in student)   # 'in' checks KEYS

None
not provided
False True


## 3. Adding, updating, deleting

In [5]:
student = {"name": "Alice", "age": 21, "grade": "A"}
student["age"] = 22               # update an existing key
student["city"] = "New York"      # add a new key
print(student)
student.update({"age": 23, "email": "alice@example.com"})   # many at once
print(student)
removed = student.pop("grade")    # remove & return the value
print(removed, student)
print(student.pop("missing", "default"))   # no error with a default
last = student.popitem()          # remove & return the LAST inserted pair
print(last)
del student["city"]
print(student)
student.clear()
print(student)

{'name': 'Alice', 'age': 22, 'grade': 'A', 'city': 'New York'}
{'name': 'Alice', 'age': 23, 'grade': 'A', 'city': 'New York', 'email': 'alice@example.com'}
A {'name': 'Alice', 'age': 23, 'city': 'New York', 'email': 'alice@example.com'}
default
('email', 'alice@example.com')
{'name': 'Alice', 'age': 23}
{}


### `setdefault`: get a value, inserting a default if missing

In [6]:
inventory = {"apples": 5}
inventory.setdefault("apples", 0)     # exists → unchanged
inventory.setdefault("pears", 0)      # missing → inserted
print(inventory)

{'apples': 5, 'pears': 0}


## 4. Views: `keys()`, `values()`, `items()`

These return **view objects** that reflect changes to the dict *live* (📗 Advanced Dictionaries):

In [7]:
d = {"k1": 1, "k2": 2}
keys_view = d.keys()
print(keys_view, d.values(), d.items())
d["k3"] = 3
print(keys_view)                   # updated automatically!
print(list(d.items()))

dict_keys(['k1', 'k2']) dict_values([1, 2]) dict_items([('k1', 1), ('k2', 2)])
dict_keys(['k1', 'k2', 'k3'])
[('k1', 1), ('k2', 2), ('k3', 3)]


## 5. Iterating

In [8]:
prices = {"apple": 0.5, "banana": 0.25, "cherry": 3.0}
for key in prices:                     # iterates over keys
    print(key)
for value in prices.values():
    print(value)
for fruit, price in prices.items():    # ✅ most common
    print(f"{fruit:<8} ₹{price:.2f}")

apple
banana
cherry
0.5
0.25
3.0
apple    ₹0.50
banana   ₹0.25
cherry   ₹3.00


❌ Don't change the size of a dict while iterating over it; iterate over a copy instead:

In [9]:
scores = {"a": 50, "b": 20, "c": 90}
for name in list(scores):        # list(...) makes a snapshot of the keys
    if scores[name] < 40:
        del scores[name]
print(scores)

{'a': 50, 'c': 90}


## 6. Sorting a dictionary

In [10]:
prices = {"cherry": 3.0, "apple": 0.5, "banana": 0.25}
print(sorted(prices))                                    # sorted keys
print(dict(sorted(prices.items())))                      # by key
print(dict(sorted(prices.items(), key=lambda kv: kv[1])))  # by value
print(max(prices, key=prices.get))                       # key with the largest value

['apple', 'banana', 'cherry']
{'apple': 0.5, 'banana': 0.25, 'cherry': 3.0}
{'banana': 0.25, 'apple': 0.5, 'cherry': 3.0}
cherry


## 7. Nesting

In [11]:
d = {"key1": {"nestkey": {"subnestkey": "value"}}}
print(d["key1"]["nestkey"]["subnestkey"])

company = {
    "name": "Acme",
    "employees": [
        {"name": "Asha", "skills": ["python", "sql"]},
        {"name": "Ravi", "skills": ["java"]},
    ],
}
for emp in company["employees"]:
    print(emp["name"], "→", ", ".join(emp["skills"]))

value
Asha → python, sql
Ravi → java


## 8. Merging dictionaries 🆕 (3.9+)

In [12]:
defaults = {"theme": "light", "lang": "en", "font": 12}
user = {"theme": "dark", "font": 14}
print(defaults | user)          # new dict; right side wins on conflicts
settings = defaults.copy()
settings |= user                # update in place
print(settings)
print({**defaults, **user})     # older (3.5+) unpacking style, same result

{'theme': 'dark', 'lang': 'en', 'font': 14}
{'theme': 'dark', 'lang': 'en', 'font': 14}
{'theme': 'dark', 'lang': 'en', 'font': 14}


## 9. Dictionary comprehensions

In [13]:
squares = {x: x ** 2 for x in range(5)}
print(squares)
words = ["apple", "kiwi", "banana"]
print({w: len(w) for w in words})
print({k: v for k, v in squares.items() if v % 2 == 0})
inverted = {v: k for k, v in {"a": 1, "b": 2}.items()}
print(inverted)

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
{'apple': 5, 'kiwi': 4, 'banana': 6}
{0: 0, 2: 4, 4: 16}
{1: 'a', 2: 'b'}


## 10. Common patterns

### Counting

In [14]:
text = "the quick brown fox jumps over the lazy dog the end"
counts = {}
for word in text.split():
    counts[word] = counts.get(word, 0) + 1
print(counts)

from collections import Counter      # the pro way (Module 12)
print(Counter(text.split()).most_common(2))

{'the': 3, 'quick': 1, 'brown': 1, 'fox': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1, 'end': 1}
[('the', 3), ('quick', 1)]


### Grouping

In [15]:
people = [("Asha", "Pune"), ("Ravi", "Delhi"), ("Meena", "Pune")]
by_city = {}
for name, city in people:
    by_city.setdefault(city, []).append(name)
print(by_city)

{'Pune': ['Asha', 'Meena'], 'Delhi': ['Ravi']}


### Lookup tables instead of long `if/elif` chains

In [16]:
def add(a, b): return a + b
def sub(a, b): return a - b
operations = {"+": add, "-": sub, "*": lambda a, b: a * b}
print(operations["*"](6, 7))

42


## 11. 🆕 `frozendict`: an immutable dictionary (Python 3.15, PEP 814)

Before 3.15, there was no built-in immutable dict. A `frozendict`:
- can't be modified after creation
- is **hashable** (if its keys and values are), so it can be a dict key or set member
- compares equal regardless of insertion order (like dict)
- is **not** a subclass of `dict` (but it is a `collections.abc.Mapping`)

In [17]:
config = frozendict(host="localhost", port=8080)
print(config, config["port"], config.get("debug", False))
print(frozendict(y=2, x=1) == frozendict(x=1, y=2))
cache = {config: "connection #1"}      # usable as a key!
print(cache[frozendict(port=8080, host="localhost")])
print(config | {"port": 9090})         # merging returns a NEW frozendict

frozendict({'host': 'localhost', 'port': 8080}) 8080 False
True
connection #1
frozendict({'host': 'localhost', 'port': 9090})


In [18]:
config["port"] = 9090      # ❌ TypeError: 'frozendict' object does not support item assignment

TypeError: 'frozendict' object does not support item assignment

> ⚠️ `x |= y` on a frozendict doesn't mutate it; it rebinds the name to a **new** frozendict (just like `+=` on a tuple).
>
> On Python < 3.15, use `types.MappingProxyType(d)` for a read-only *view* of a dict.

## 📝 Summary

| Task | Code |
|---|---|
| Read safely | `d.get(k, default)` |
| Add/update | `d[k] = v`, `d.update(...)`, `d \|= other` |
| Remove | `d.pop(k[, default])`, `del d[k]`, `popitem()`, `clear()` |
| Iterate | `for k, v in d.items()` |
| Merge | `a \| b` |
| Insert-if-missing | `d.setdefault(k, default)` |
| Immutable | 🆕 `frozendict(...)` |

## 🏋️ Practice (📗 assessment style)
1. Using keys and indexing, grab `"hello"` from `d = {'simple_key': 'hello'}`.
2. Grab `"hello"` from `d = {'k1': {'k2': 'hello'}}`.
3. Grab `"hello"` from `d = {'k1': [{'nest_key': ['this is deep', ['hello']]}]}`.
4. Count the frequency of each character in `"mississippi"` (without `Counter`).
5. Invert `{"a": 1, "b": 2, "c": 1}` into `{1: ["a", "c"], 2: ["b"]}`.
6. Merge `{"x": 1, "y": 2}` and `{"y": 20, "z": 30}` so the second wins.

<details><summary>Show solutions</summary>

```python
print({'simple_key': 'hello'}['simple_key'])
print({'k1': {'k2': 'hello'}}['k1']['k2'])
d = {'k1': [{'nest_key': ['this is deep', ['hello']]}]}
print(d['k1'][0]['nest_key'][1][0])

freq = {}
for ch in "mississippi":
    freq[ch] = freq.get(ch, 0) + 1

inv = {}
for k, v in {"a": 1, "b": 2, "c": 1}.items():
    inv.setdefault(v, []).append(k)

print({"x": 1, "y": 2} | {"y": 20, "z": 30})
```
</details>